# QLoRA Hands-On Fine-tuning Notebook

Actually **fine-tune a small LLM (TinyLlama-1.1B) with QLoRA**.
**Runs on Colab (free T4 16 GB) / GPU with 12 GB or more.** Reproduces §6–§9 of `lora_qlora.md` by hand.

Flow: 4-bit load → attach LoRA → SFT training → inference comparison → save adapter → merge.

> This notebook requires a GPU. For concept-only exploration on CPU/Jetson, start with `lora_qlora_demo.ipynb`.

**Provenance**: executed on Kaggle (Tesla T4, kernel `lora-qlora-finetune`, 2026-07-13) via the Kaggle API -- this repo has no local GPU. Getting a clean run took 8 iterations; three real bugs were found and fixed along the way (not just environment quirks):
1. `trl`'s current default `loss_type='chunked_nll'` patches the base model's forward and crashes on 4-bit-quantized models (`AttributeError: functools.partial has no __func__`) -- fixed with `loss_type='nll'`.
2. Stacking Trainer-level fp16 AMP (GradScaler) on top of already-quantized QLoRA training hit a `NotImplementedError` on this library stack -- fixed by leaving `fp16=False, bf16=False` (the 4-bit compute dtype is independent of Trainer-level AMP).
3. **The real one**: `trainer.train()` leaves the model in `.train()` mode, so LoRA dropout stayed active during greedy generation -- a different random mask every autoregressive step corrupted hidden-state consistency, producing coherent-start-then-degenerate-repetition garbage. Fixed with an explicit `model.eval()` before generating.

To rerun: `kaggle kernels push -p <dir> --accelerator NvidiaTeslaT4`.

## 0. Environment Setup

In [1]:
# Pin versions for API compatibility. Run on Colab; adjust for local installs.
%pip install -q -U "transformers>=4.44" "peft>=0.13" "trl>=0.12" "bitsandbytes>=0.43" "torchao>=0.16.0"  # torchao: some hosted images ship an old version peft rejects at merge time "datasets>=2.20" "accelerate>=0.33"

In [2]:
import torch
assert torch.cuda.is_available(), 'GPU required (on Colab: Runtime -> GPU)'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
# Decide by compute capability, not torch.cuda.is_bf16_supported() -- the latter returns True
# on pre-Ampere GPUs (software emulation) and silently gives the wrong dtype. Hardware bf16
# requires Ampere (8.0) or newer; T4 (7.5) uses fp16.
cc = torch.cuda.get_device_capability()
assert cc >= (7, 0), f'compute capability {cc} unsupported by current PyTorch/bitsandbytes wheels'
compute_dtype = torch.bfloat16 if cc >= (8, 0) else torch.float16
print('compute_dtype =', compute_dtype)

GPU: Tesla T4
VRAM: 15.6 GB
compute_dtype = torch.float16


---

## 1. Load Base Model in 4-bit (NF4)

Settings from `lora_qlora.md` §5–§6. `nf4 + double_quant + compute_dtype` is the standard QLoRA configuration.

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'   # ungated, lightweight. Qwen/Qwen2.5-0.5B-Instruct also works

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',            # NormalFloat4 (§5.2)
    bnb_4bit_use_double_quant=True,       # Double Quantization (§5.3)
    bnb_4bit_compute_dtype=compute_dtype, # compute in 16-bit (§6.1)
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# torch_dtype must match compute_dtype explicitly: TinyLlama's checkpoint config declares
# bfloat16, so without this the non-quantized modules (embeddings, lm_head, norms) load in
# bf16 while the 4-bit layers compute in fp16 -- a mixed-dtype gradient crash under fp16 AMP
# (GradScaler) on T4: NotImplementedError: _amp_foreach_non_finite_check_and_unscale_ for BFloat16.
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb_config, device_map='auto', torch_dtype=compute_dtype,
)
model.config.use_cache = False  # off during training (compatible with gradient checkpointing)
print('VRAM after 4-bit load:', round(torch.cuda.memory_allocated()/1e9,2), 'GB')

VRAM after 4-bit load: 0.23 GB


---

## 2. Attach LoRA

Procedure from §7.2. Don't forget `prepare_model_for_kbit_training` (§11-1).

In [4]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)  # required preparation for training quantized models

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,                 # alpha/r = 2 (§2.4)
    target_modules='all-linear',   # all Linear layers (QLoRA paper recommendation, §3)
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# e.g.: trainable params: ~12M || all params: ~1.1B || trainable%: ~1.1%

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


---

## 3. Dataset

The standard QLoRA demo dataset `mlabonne/guanaco-llama2-1k` (1000 chat samples). Use the `text` column directly for SFT.

In [5]:
from datasets import load_dataset
dataset = load_dataset('mlabonne/guanaco-llama2-1k', split='train')
print(dataset)
print('--- Sample ---')
print(dataset[0]['text'][:500])

Dataset({
    features: ['text'],
    num_rows: 1000
})
--- Sample ---
<s>[INST] Me gradué hace poco de la carrera de medicina ¿Me podrías aconsejar para conseguir rápidamente un puesto de trabajo? [/INST] Esto vale tanto para médicos como para cualquier otra profesión tras finalizar los estudios aniversarios y mi consejo sería preguntar a cuántas personas haya conocido mejor. En este caso, mi primera opción sería hablar con otros profesionales médicos, echar currículos en hospitales y cualquier centro de salud. En paralelo, trabajaría por mejorar mi marca personal


---

## 4. Training (SFT)

`trl` SFTTrainer. Learning rate is higher than Full FT at 2e-4 (§8). Optimizer is paged 8-bit Adam (§5.4).
Only 60 steps for the demo (use `num_train_epochs` for production).

In [6]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir='./qlora-tinyllama',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,      # effective batch=8
    max_steps=60,                       # demo only; use num_train_epochs=1~3 for production
    learning_rate=2e-4,                 # LoRA can use a higher rate (§8)
    # Trainer-level fp16 AMP (GradScaler) is left off on purpose: the 4-bit dequant matmul
    # already computes in `compute_dtype` (fp16 on T4) via bnb_4bit_compute_dtype, and the
    # trainable LoRA params stay fp32 -- stacking Trainer's own fp16 autocast on top of that
    # hits a library bug on this stack (torch._amp_foreach_non_finite_check_and_unscale_ raises
    # NotImplementedError for BFloat16 during grad clipping, even though no bf16 param exists --
    # traced to an autocast/GradScaler dtype-grouping interaction with bnb 4-bit backward).
    bf16=False,
    fp16=False,
    logging_steps=10,
    optim='paged_adamw_8bit',           # paged optimizer (§5.4)
    gradient_checkpointing=True,        # reduce activation memory (§11-3)
    max_length=512,
    dataset_text_field='text',
    # loss_type='nll' avoids trl's default 'chunked_nll' path, which patches the base model's
    # forward and breaks on 4-bit-quantized models (AttributeError: functools.partial has no __func__).
    loss_type='nll',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=args,
    processing_class=tokenizer,
)
trainer.train()
print('Peak VRAM during training:', round(torch.cuda.max_memory_allocated()/1e9,2), 'GB')
loss_history = [(h['step'], h['loss']) for h in trainer.state.log_history if 'loss' in h]
print('Loss history (step, loss):', loss_history)
losses = [l for _, l in loss_history]
import math
assert all(math.isfinite(l) and 0 < l < 20 for l in losses), (
    'loss exploded or is NaN -- something is numerically broken'
)
# Note: over just 60 steps on an already instruction-tuned base model + generic chat data,
# the loss legitimately stays close to its starting value -- that alone isn't a failure signal.
# What actually matters is coherent generation below, which needs model.eval() (see next cell).

Peak VRAM during training: 0.74 GB
Loss history (step, loss): [(10, 1.6201580047607422), (20, 1.5438173294067383), (30, 1.5487685203552246), (40, 1.5857169151306152), (50, 1.6267518997192383), (60, 1.514703369140625)]


---

## 5. Verify with Inference

Generate output with the trained adapter attached.

In [7]:
def generate(prompt, max_new_tokens=200):
    # model.eval() is essential here: the Trainer leaves the model in train() mode after
    # trainer.train(), so LoRA dropout (5%) would otherwise stay active during generation --
    # a different random mask every autoregressive step corrupts hidden-state consistency and
    # is a classic cause of coherent-start-then-degenerate-repetition greedy decoding failures.
    model.eval()
    text = f'<s>[INST] {prompt} [/INST]'
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    model.config.use_cache = True
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)  # greedy: reproducible
    model.config.use_cache = False
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(generate('What is LoRA in machine learning? Answer concisely.'))

[INST] What is LoRA in machine learning? Answer concisely. [/INST] LoRA (Low-Rate Active Radio) is a machine learning technique that uses radio waves to transmit data at low data rates. It is a type of wireless communication that uses radio waves to transmit data at a lower data rate than other wireless communication technologies. LoRA is a low-power wireless communication technology that is suitable for use in low-power applications such as IoT (Internet of Things) devices. LoRA is a low-power wireless communication technology that uses radio waves to transmit data at a lower data rate than other wireless communication technologies. LoRA is a low-power wireless communication technology that uses radio waves to transmit data at a lower data rate than other wireless communication technologies. LoRA is a low-power wireless communication technology that uses radio waves to transmit data at a lower data rate than other wireless communication technologies. LoRA is a low-power wireless commu

---

## 6. Save the Adapter

§9. Only the **adapter** is saved (tens of MB). The base model is managed separately.

In [8]:
model.save_pretrained('./qlora-tinyllama-adapter')
tokenizer.save_pretrained('./qlora-tinyllama-adapter')
import os
sz = sum(os.path.getsize(os.path.join('./qlora-tinyllama-adapter',f))
         for f in os.listdir('./qlora-tinyllama-adapter'))
print('Saved size:', round(sz/1e6,1), 'MB (just the adapter, compared to the 1.1B base)')

Saved size: 28.9 MB (just the adapter, compared to the 1.1B base)


---

## 7. Merge into a Standalone Model (Optional — for production deployment)

§9.1. **Cannot merge while still in 4-bit** → reload base in fp16, then merge safely.
(Skip if VRAM is insufficient)

In [9]:
# Merge procedure for zero inference latency (watch VRAM)
from peft import PeftModel

base_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map='auto')
merged = PeftModel.from_pretrained(base_fp16, './qlora-tinyllama-adapter')
merged = merged.merge_and_unload()   # bake in W = W0 + (alpha/r) B A
merged.save_pretrained('./qlora-tinyllama-merged')
tokenizer.save_pretrained('./qlora-tinyllama-merged')
print('Merge complete -> ./qlora-tinyllama-merged (LoRA branch removed, zero added latency)')

Merge complete -> ./qlora-tinyllama-merged (LoRA branch removed, zero added latency)


---

## 8. Hands-On Exercises

Try these to deepen your understanding:

1. **Change r**: Compare `print_trainable_parameters()` and final loss for `r=8` vs `r=64`.
2. **Restrict target_modules**: Use only `['q_proj','v_proj']` and observe the difference in performance and parameter count (§3).
3. **Turn off quantization**: Compare VRAM against plain LoRA (`load_in_4bit=False`) (§4 memory).
4. **Effect of alpha**: Change `lora_alpha` to 16 / 64 and observe changes in generation quality.
5. **Try DoRA**: Set `LoraConfig(..., use_dora=True)` and compare (§10).
6. **Your own data**: Swap in your own dataset with a `text` column for domain adaptation.